# xG — Copa do Mundo 2022

Modelo de *expected goals*: dado um chute, qual a probabilidade de virar gol.

Dados: [StatsBomb Open Data](https://github.com/statsbomb/open-data) — 64 partidas.

In [1]:
import os, json, time
import requests

BASE = 'https://raw.githubusercontent.com/statsbomb/open-data/master/data'
COMPETICAO, TEMPORADA = 43, 106   # 43 = FIFA World Cup, 106 = 2022

CACHE = 'statsbomb_cache'
os.makedirs(CACHE, exist_ok=True)

In [2]:
def baixar(caminho):
    """Baixa data/<caminho>. Se ja estiver em disco, le do cache."""
    local = os.path.join(CACHE, caminho.replace('/', '_'))
    if os.path.exists(local):
        return json.load(open(local, encoding='utf-8'))

    dados = requests.get(f'{BASE}/{caminho}', timeout=30).json()
    json.dump(dados, open(local, 'w', encoding='utf-8'))
    time.sleep(0.2)   # 64 requisicoes seguidas, nao vale martelar o servidor
    return dados


partidas = baixar(f'matches/{COMPETICAO}/{TEMPORADA}.json')
for p in partidas:
    baixar(f'events/{p["match_id"]}.json')

print(len(partidas), 'partidas no cache')

64 partidas no cache


O cache bruto tem ~150 MB e fica fora do Git. O `extrair_chutes.py` percorre esses arquivos, filtra os eventos de chute e gera o `chutes_wc22.parquet` (0,12 MB), que é o que o resto do notebook usa.

In [3]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_parquet('chutes_wc22.parquet')
df.shape

(1494, 27)

In [5]:
df.head()

,match_id,shot_id,period,minute,second,x,y,duration,under_pressure,statsbomb_xg,...,shot_technique,shot_body_part,first_time,one_on_one,aerial_won,open_goal,deflected,follows_dribble,saved_to_post,gol
0,3857254,14374288-0565-4f12-b39e-318100137d0a,1,2,2,92.6,52.0,0.338598,False,0.023820,...,Normal,Right Foot,False,False,False,False,False,False,False,0
1,3857254,89cbea71-b5ef-4c16-9bd3-39b511619958,1,4,10,114.0,54.8,0.105592,False,0.014060,...,Normal,Left Foot,False,False,False,False,False,False,False,0
2,3857254,700d58f9-8032-4543-9a9d-b134c6996608,1,10,46,93.4,44.5,0.136106,False,0.033115,...,Normal,Right Foot,False,False,False,False,False,False,False,0
3,3857254,63422fb4-45a3-4f30-8340-53d468a8b8cb,1,11,47,114.7,29.6,0.970357,False,0.043661,...,Normal,Head,False,False,False,False,False,False,False,0
4,3857254,490287f7-4b99-49f8-ac89-16766ac7a05b,1,22,5,115.3,32.5,0.013816,False,0.124033,...,Volley,Right Foot,True,False,False,False,False,False,False,0


In [6]:
df.columns

Index(['match_id', 'shot_id', 'period', 'minute', 'second', 'x', 'y',
       'duration', 'under_pressure', 'statsbomb_xg', 'player', 'team',
       'possession_team', 'play_pattern', 'position', 'shot_type',
       'shot_outcome', 'shot_technique', 'shot_body_part', 'first_time',
       'one_on_one', 'aerial_won', 'open_goal', 'deflected', 'follows_dribble',
       'saved_to_post', 'gol'],
      dtype='object')

In [7]:
df.head().T

,0,1,2,3,4
match_id,3857254,3857254,3857254,3857254,3857254
shot_id,14374288-0565-4f12-b39e-318100137d0a,89cbea71-b5ef-4c16-9bd3-39b511619958,700d58f9-8032-4543-9a9d-b134c6996608,63422fb4-45a3-4f30-8340-53d468a8b8cb,490287f7-4b99-49f8-ac89-16766ac7a05b
period,1,1,1,1,1
minute,2,4,10,11,22
second,2,10,46,47,5
x,92.6,114.0,93.4,114.7,115.3
y,52.0,54.8,44.5,29.6,32.5
duration,0.338598,0.105592,0.136106,0.970357,0.013816
under_pressure,False,False,False,False,False
statsbomb_xg,0.02382,0.01406,0.033115,0.043661,0.124033


In [8]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

In [9]:
df['shot_type'].unique()

array(['open_play', 'corner', 'free_kick', 'penalty'], dtype=object)

In [10]:
df.isnull().sum()

match_id           0
shot_id            0
period             0
minute             0
second             0
x                  0
y                  0
duration           0
under_pressure     0
statsbomb_xg       0
player             0
team               0
possession_team    0
play_pattern       0
position           0
shot_type          0
shot_outcome       0
shot_technique     0
shot_body_part     0
first_time         0
one_on_one         0
aerial_won         0
open_goal          0
deflected          0
follows_dribble    0
saved_to_post      0
gol                0
dtype: int64

In [11]:
import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline

In [12]:
chutes = df[~df.shot_type.isin(['penalty', 'corner'])].copy() #Removendo do df, chutes de penalty e escanteios

## Geometria do chute

As duas features mais importantes de um modelo de xG não existem no dado: são derivadas de `x` e `y`.

O campo do StatsBomb é 120 x 80, com o gol adversário em $x = 120$ entre $y = 36$ e $y = 44$.

**Distância** é Pitágoras até o centro do gol. **Ângulo** é a abertura que o chute enxerga entre as duas traves: quanto maior, mais gol disponível para mirar.

In [13]:
chutes.nunique()

match_id             64
shot_id            1428
period                4
minute              118
second               60
x                   327
y                   394
duration           1427
under_pressure        2
statsbomb_xg       1428
player              429
team                 32
possession_team      32
play_pattern          9
position             22
shot_type             2
shot_outcome          8
shot_technique        7
shot_body_part        4
first_time            2
one_on_one            2
aerial_won            2
open_goal             2
deflected             2
follows_dribble       2
saved_to_post         2
gol                   2
dtype: int64

In [14]:
#Transformando a distância até o meio do gol e o ângulo de abertura entre as traves
#Primeiramente no dataset, o campo é 120 x 80, com o gol do oponente em x = 120, y = 36 entre y = 44
#A distância é definida pelo teorema de pitágoras, até o gol
#Ja o ângulo é abertura que o jogador enxerga entre as traves durante o chute, quanto maior o ângulo mais o jogador vê o gol

In [15]:
GOAL_X = 120.0        
GOAL_CENTER = 40.0    
GOAL_HALF = 4.0      


def shot_geometry(x, y, degrees=True):
    
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    dx = GOAL_X - x
    dist = np.hypot(dx, y - GOAL_CENTER)

    cross = 2.0 * GOAL_HALF * dx                  # a x b
    dot   = dist**2 - GOAL_HALF**2                # a . b
    ang   = np.arctan2(cross, dot)

    return dist, (np.degrees(ang) if degrees else ang)

In [16]:
chutes['distancia'], chutes['angulo'] = shot_geometry(chutes.x, chutes.y)

chutes[['x', 'y', 'distancia', 'angulo', 'gol']].head()

,x,y,distancia,angulo,gol
0,92.6,52.0,29.912539,14.006170,0
1,114.0,54.8,15.969972,11.354176,0
2,93.4,44.5,26.977954,16.644360,0
3,114.7,29.6,11.672618,19.422587,0
4,115.3,32.5,8.850989,31.096029,0


In [29]:
#Framework de validação

In [30]:
from sklearn.model_selection import train_test_split

In [31]:
df_full_train, df_test = train_test_split(chutes, test_size = 0.2, random_state = 1)

In [32]:
df_train, df_val = train_test_split(df_full_train, test_size = 0.25, random_state = 1)

In [33]:
len(df_train), len(df_val)

(856, 286)

In [34]:
df_train = df_train.reset_index(drop = True)
df_val = df_val.reset_index(drop = True)
df_test = df_test.reset_index(drop = True)

In [35]:
y_train = df_train.gol.values
y_val = df_val.gol.values
y_test = df_test.gol.values

In [36]:
df_test.head().T

,0,1,2,3,4
match_id,3857279,3857265,3857283,3857263,3857297
shot_id,afced147-4cd3-47fc-aef0-dea4baae4623,2a275f0d-eb16-4923-9c98-b2e7f7fdf086,71c9aaf5-f70c-4073-b471-3f66683ea104,a21bf591-0552-4c0e-8cb5-f284ab8ff959,2be63b79-06c8-46a1-8688-d5257673d4d9
period,1,2,1,1,1
minute,44,59,34,35,12
second,38,13,54,4,17
x,112.9,107.9,94.1,109.8,105.5
y,40.2,29.9,42.1,54.6,54.5
duration,0.18499,0.131516,0.009438,0.020692,0.62361
under_pressure,False,False,True,False,False
statsbomb_xg,0.551751,0.060923,0.021632,0.04777,0.038397
